# 🧩 **CellPatcher strategy examples**
___

This notebook shows what each **patching strategy** of CellPatcher gives, on one image of your
experiment. It patches the image in memory and **writes nothing**: use it to choose a strategy and
its settings, then export the patches with the **CellPatcher notebook**, which gives exactly the
patches shown here for the same settings.

Like the CellPatcher notebook, it reads every image and mask **through the run of the MorphoBricks
preprocessing notebook** (version 0.4): you point it to the preprocessing run's output folder.

| Strategy | What you get | Needs masks? |
|---|---|---|
| **Cell** | One patch per segmented object (cell, nucleus, ...) | Yes |
| **Grid** | Regular tiles covering the image | No |
| **Random** | Patches at random positions (reproducible with a seed) | Optional |
| **Whole FOV** | One patch per image | No |

### Quick workflow

1. Run the **Step 0** cell to start.
2. **Step 1:** paste the preprocessing output folder, press **Check the preprocessing run** and
   choose the **Example image**.
3. Run each strategy's cell, change its settings if you like, and press its **Show the ... example**
   button.

### Before you start

- [ ] You ran the **MorphoBricks preprocessing notebook** (version 0.4) on your experiment, with
  your masks folder for the cell strategy, and it finished. Its output folder holds
  **run_manifest.json** and **plane_histograms.csv**.

> 🛡️ **Your images are never changed, and nothing is written.**

___
## 🧰 **Step 0: Start the notebook**

Click the cell below and press **Shift + Enter** (or the ▶ button in the toolbar). A green box
appears when everything is ready.

> **Colab users:** Step 0 connects your Google Drive (allow access in the window that opens)
> and installs CellPatcher **v0.2.0rc3** from GitHub, which takes a minute or two. Keep the
> preprocessing output folder and your experiment on Google Drive: in this notebook your Drive
> is the folder **/content/drive/MyDrive**. Everything else on the runtime is wiped when it
> disconnects; then run Step 0 again.
>
> **JupyterLab users:** Step 0 uses CellPatcher as installed for this notebook's kernel (or the
> copy of the repository around the notebook), installs what is missing, and prints the Python
> version and working folder. Nothing is mounted.

> 💡 **Tip:** the code of each cell is hidden; you only need to run the cells, from top to bottom,
> one after the other.

In [ ]:
# @title Run to start the CellPatcher strategy examples
import platform
import subprocess
import sys
from importlib import import_module, invalidate_caches
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from pathlib import Path

# The notebook's version and name, as LabConstrictor's version check reads them
# (notebooks/notebook_latest_versions.yaml lists the latest version).
current_version = "0.2.0rc3"
notebook_name = "CellPatcher_Strategy_Examples"
# The release installed from GitHub when the package is missing (Google Colab, or a
# downloaded notebook). Set to the new tag, with current_version, before each release is
# tagged, so a tagged notebook installs its own release.
RELEASE_TAG = "v0.2.0rc3"
REPOSITORY = "https://github.com/CellMigrationLab/CellPatcher.git"
# The package's extras to install; on Colab also zarr, to read OME-Zarr images.
EXTRAS = "notebook"
# The notebook's interface in the package, as "module:class".
INTERFACE = "cellpatcher.notebook_examples:StrategyExamplesNotebook"

if "google.colab" in sys.modules:
    # Colab only: Google Drive holds the data and the results, which outlive the runtime.
    from google.colab import drive, output

    print(
        "Google Colab: connecting Google Drive. Allow access in the window that opens; "
        "your Drive is then the folder /content/drive/MyDrive.",
        flush=True,
    )
    try:
        drive.mount("/content/drive")
    except Exception as exc:  # a refused or interrupted authorisation
        raise RuntimeError(
            "Google Drive could not be connected. Run Step 0 again and allow access to "
            "your Drive; your data and output folders must be on Drive."
        ) from exc
    # Colab ships ipywidgets 7; this notebook's widgets need ipywidgets 8's front end.
    output.enable_custom_widget_manager()
    EXTRAS += ",zarr"
else:
    print(
        f"Local Jupyter: Python {platform.python_version()} on {platform.platform()}\n"
        f"Working folder: {Path.cwd()}",
        flush=True,
    )


def _missing_requirements() -> list[str]:
    """Return the notebook's requirements this kernel cannot import (or too old)."""
    required = (
        "cellpatcher",
        "morphobricks_preprocessing",
        "morphobricks_normalization",
        "numpy",
        "tifffile",
        "ipywidgets",
        "matplotlib",
        *(extra for extra in ("zarr", "torch") if extra in EXTRAS.split(",")),
    )
    missing = [name for name in required if find_spec(name) is None]
    try:
        widgets_major = int(version("ipywidgets").split(".")[0])
    except (PackageNotFoundError, ValueError):
        widgets_major = 8
    if widgets_major < 8 and "ipywidgets" not in missing:
        missing.append("ipywidgets 8")
    return missing


def _install_target() -> tuple[list[str], str]:
    """Return what pip installs when something is missing, and where CellPatcher comes from."""
    current = Path.cwd().resolve()
    checkout = next(
        (
            folder
            for folder in (current, *current.parents)
            if (folder / "setup.py").is_file() and (folder / "src/cellpatcher/__init__.py").is_file()
        ),
        None,
    )
    if checkout is not None:
        source = checkout / "src"
        loaded = sys.modules.get("cellpatcher")
        if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(source):
            raise RuntimeError(
                "This kernel has already loaded another copy of CellPatcher. "
                "Restart the kernel, then run Step 0 to use this checkout."
            )
        # A local source tree works immediately, including in a git worktree.
        sys.path[:] = [str(source), *(entry for entry in sys.path if entry != str(source))]
        invalidate_caches()
        return ["-e", f"{checkout}[{EXTRAS}]"], f"the local checkout at {checkout}"
    if find_spec("cellpatcher") is not None:
        try:
            return [f"cellpatcher[{EXTRAS}]"], f"the installed package, version {version('cellpatcher')}"
        except PackageNotFoundError:
            return [f"cellpatcher[{EXTRAS}]"], "the installed package"
    return [f"cellpatcher[{EXTRAS}] @ git+{REPOSITORY}@{RELEASE_TAG}"], f"release {RELEASE_TAG} from GitHub"


def _start_notebook() -> object:
    """Prepare this kernel's environment and return the notebook interface."""
    # This standalone cell can run before the package's Python requirement is checked.
    if sys.version_info < (3, 11):  # noqa: UP036
        raise RuntimeError("Select a Python 3.11 or newer notebook kernel, then run Step 0 again.")

    environment = "Google Colab" if "google.colab" in sys.modules else "local Jupyter"
    print(f"Checking the {environment} environment...", flush=True)
    install_target, origin = _install_target()
    missing = _missing_requirements()
    if missing:
        print(f"Installing notebook requirements ({', '.join(missing)})...", flush=True)
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *install_target],
                check=True,
            )
        except (subprocess.CalledProcessError, OSError) as exc:
            raise RuntimeError(
                "Notebook setup could not install its requirements. Check the installation "
                "output above, internet access and write permissions, then rerun Step 0. "
                "Installing from GitHub also needs Git."
            ) from exc
        invalidate_caches()
        missing = _missing_requirements()
        if missing:
            raise RuntimeError(
                f"Installation finished, but this kernel still cannot find {', '.join(missing)}. "
                "Restart the kernel and run Step 0 again."
            )
    loaded_widgets = str(getattr(sys.modules.get("ipywidgets"), "__version__", "8")).split(".")[0]
    if loaded_widgets.isdigit() and int(loaded_widgets) < 8:
        raise RuntimeError(
            "This kernel loaded ipywidgets 7 before ipywidgets 8 was installed. Restart the "
            "kernel (on Colab: Runtime > Restart session), then run Step 0 again."
        )

    print(f"Using {origin}.\nNotebook {notebook_name} {current_version}; Python: {sys.executable}", flush=True)
    module, _, name = INTERFACE.partition(":")
    try:
        interface = import_module(module)
    except ImportError as exc:
        raise RuntimeError(
            "CellPatcher was found, but the notebook interface could not be loaded. "
            "Read the original error below. If packages were changed in this session, "
            "restart the kernel and run Step 0 again."
        ) from exc
    return getattr(interface, name)()


notebook = _start_notebook()
notebook.welcome()

___
## 📂 **Step 1: Choose the preprocessing run and the example image**

> ⚠️ **Run preprocessing first.** CellPatcher needs the output folder of a finished run of the
> **MorphoBricks preprocessing notebook** (version 0.4) or of its command line
> (`morphobricks-preprocessing check`, then `design`, then `run`).

Run the cell below, then:

1. Paste the path of the **preprocessing output folder** into **Preprocessing output folder**, or
   click **Browse…**.
2. Only if you **moved or copied** the experiment (or its masks folder) since you preprocessed it,
   tick **The experiment was moved or copied since preprocessing** and give where they are now.
3. Press **Check the preprocessing run**. A green card lists the images, conditions, channels and
   mask types; a red card says what to do.
4. Choose the **Example image**: every example below patches this image.

> **Colab users:** paste paths that start with **/content/drive/MyDrive/**. **Browse…** cannot
> open a dialog on Colab: open the **Files** panel in the left bar, right-click a folder, choose
> **Copy path** and paste it.
>
> **JupyterLab users:** paste the path or use **Browse…**, which opens your computer's folder
> dialog.

In [ ]:
# @title Run to choose the preprocessing run
notebook.run_step()

___
## 🔬 **Cell strategy: one patch per object**

One patch per segmented object of the run's masks, centred on the object. Settings:

- **Mask type**: which of the run's masks holds the objects.
- **Patch size**: per object (the object's own box), fixed, or a statistic of the objects of the
  whole dataset (press **Scan the object sizes of the dataset** first).
- **Skip objects touching the image border**, and the object area limits.
- **Blank the pixels outside the object (apply mask)**.

Masks hold instance labels (each object its own number). Boolean masks need
**Masks are boolean: label their connected components**. Run the cell, then press
**Show the Cell example**.

In [ ]:
# @title Run to show the cell strategy
notebook.strategy_step("cell")

___
## 🔲 **Grid strategy: regular tiles**

Square tiles of **Tile size (pixels)**, with an **Overlap (fraction, 0 to <1)** between
neighbours, and optionally **Keep partial tiles at the edges**. No mask is needed. Run the cell,
then press **Show the Grid example**. If the image is smaller than one tile, there are no patches:
lower the tile size.

In [ ]:
# @title Run to show the grid strategy
notebook.strategy_step("grid")

___
## 🎲 **Random strategy: patches at random positions**

**Patches per image** squares of **Patch size (pixels)** at random positions. The same
**Seed (reproducible sampling)** gives the same positions every time, so a run is reproducible.
Tick **Sample only inside the masks** to keep the patches on the objects of the **Mask type**. Run the cell, then press **Show the Random example**.

In [ ]:
# @title Run to show the random strategy
notebook.strategy_step("random")

___
## 🖼️ **Whole FOV strategy: the whole image**

One patch holding the whole image, for models that look at the whole field of view. It has no
settings. Run the cell, then press **Show the Whole FOV example**.

In [ ]:
# @title Run to show the whole FOV strategy
notebook.strategy_step("whole_fov")

___
## ✅ **After running**

- [ ] Choose the strategy and settings that give the patches you need.
- [ ] Open the **CellPatcher notebook**, check the same preprocessing run, choose the same strategy
  and settings in its step 2 and export the patches in its step 4. With the same settings its
  preview shows the same patches as these examples; add normalization there if you need it (the
  patch scope leaves out patches whose own pixels have no spread).

More help: the [patching modes](https://github.com/CellMigrationLab/CellPatcher/blob/main/docs/patching_modes.md)
and the [documentation](https://github.com/CellMigrationLab/CellPatcher/blob/main/docs/README.md).